# Session 6, clustering demo

Three quick demos: **k-means** on a tiny viewer set, **cosine** on show genres, and the **penguin check**. Follow along; the hands-on is the separate *Cluster the days* notebook.

## 1 · K-means on 30 Netflix viewers

Two features only, so we can see everything: `hours_per_week`, and `pct_series`, the fraction of their viewing that is TV series rather than films (0 = only films, 1 = only series).

In [ ]:
# --- Load the data -------------------------------------------------
DATA_URL = ""   # paste a hosted CSV link, or leave blank to upload netflix_viewers_toy.csv
import pandas as pd
if DATA_URL:
    v = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload netflix_viewers_toy.csv ...")
        v = pd.read_csv(next(iter(files.upload())))
    except Exception:
        v = pd.read_csv("data/netflix_viewers_toy.csv")
v.head()

### Plot first, predict before you compute
How many groups do *you* see? Commit to a number before running anything else.

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(v['hours_per_week'], v['pct_series']); plt.xlabel('hours watched / week')
plt.ylabel('fraction that is series (0=films, 1=series)'); plt.show()

### First, let the machine try it on the raw numbers

Nothing clever, just k-means on the two columns exactly as they were recorded.

In [ ]:
from sklearn.cluster import KMeans
FEATS = ['hours_per_week', 'pct_series']
v['raw_cluster'] = KMeans(n_clusters=3, n_init=10, random_state=7).fit_predict(v[FEATS])
plt.scatter(v['hours_per_week'], v['pct_series'], c=v['raw_cluster'], cmap='viridis')
plt.xlabel('hours watched / week'); plt.ylabel('fraction that is series')
plt.title('clustered on the raw numbers'); plt.show()

**Look at what it did.** It sliced the picture into vertical bands. Viewers who watch almost nothing but series are sitting in the same group as viewers who watch almost nothing but films, purely because they happen to watch a similar number of hours. That is not what your eye saw.

### Why: one feature is measured in much bigger numbers

Distance adds the gap on each feature together. So the feature with the wider spread wins every argument.

In [ ]:
print(v[FEATS].max() - v[FEATS].min())
# The widest gap between any two viewers, on each feature.

A viewer's **entire taste**, all films at one end and all series at the other, is worth less than **one extra hour** of watching. Nobody decided hours matter more. It fell out of the units the data happened to be recorded in.

### Fix the ruler, then cluster again

`StandardScaler` rewrites each feature so both can vary by the same amount. One line.

In [ ]:
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(v[FEATS])
v['cluster'] = KMeans(n_clusters=3, n_init=10, random_state=7).fit_predict(Z)
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
ax[0].scatter(v['hours_per_week'], v['pct_series'], c=v['raw_cluster'], cmap='viridis')
ax[0].set_title('raw numbers')
ax[1].scatter(v['hours_per_week'], v['pct_series'], c=v['cluster'], cmap='viridis')
ax[1].set_title('rescaled first')
for a in ax: a.set_xlabel('hours watched / week')
ax[0].set_ylabel('fraction that is series'); plt.show()

Same data, same algorithm, same k. **Only the units changed, and the answer changed with them.** This is the single most common way a clustering goes quietly wrong.

> Ask Claude: *"Why did rescaling change which viewers ended up together here?"* Then check its answer against the plot.

### How many clusters? Elbow + silhouette
> Ask Claude: *"Plot the k-means inertia for k=1..6 and the silhouette for k=2..6 on Z."*

In [ ]:
from sklearn.metrics import silhouette_score
for k in range(2, 7):
    lab = KMeans(n_clusters=k, n_init=10, random_state=7).fit_predict(Z)
    print(f'k={k}  silhouette={silhouette_score(Z, lab):.3f}')

Both point to **k = 3** here, because this toy is deliberately clean. On the real bike data later they disagree, and then you choose.

Now the part the machine cannot do. **Name the three groups.** Profile them first, then give each a name a colleague would understand.

In [ ]:
print(v.groupby('cluster')[FEATS].mean().round(2))
print(v['cluster'].value_counts().sort_index())

One heavy-watching group is almost all series, another heavy-watching group is almost all films, and a lighter group sits in between. *Bingers*, *Film buffs*, *Casual dippers* is one reasonable naming. Yours may differ, and that is fine as long as you can defend it.

## 2 · Cosine: same direction, different size

Genres as vectors. Cosine ignores how 'big' a show is and asks which way it points.

In [ ]:
# --- Load the data -------------------------------------------------
DATA_URL = ""   # paste a hosted CSV link, or leave blank to upload netflix_shows_small.csv
import pandas as pd
if DATA_URL:
    shows = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload netflix_shows_small.csv ...")
        shows = pd.read_csv(next(iter(files.upload())))
    except Exception:
        shows = pd.read_csv("data/netflix_shows_small.csv")
shows.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
G = shows.drop(columns='title')
sim = cosine_similarity(G)
import numpy as np
i = list(shows.title).index('Stranger Things')
for j in np.argsort(-sim[i]):
    if j != i:
        print(f'{shows.title[j]:18s} cosine={sim[i][j]:.2f}')

Stranger Things sits closest to **Dark** (both sci-fi drama), well away from the kids or reality shows. This is exactly the idea behind “because you watched…”, which we build in Session 8.

## 3 · The penguin check, trust it, then let go

Cluster data where we secretly *do* know the answer, and see whether k-means recovers it.

344 penguins measured at Palmer Station, Antarctica, 2007-2009, three real species. We hide the species column, cluster on the body measurements alone, then compare.

In [ ]:
# --- Load the data -------------------------------------------------
DATA_URL = ""   # paste a hosted CSV link, or leave blank to upload palmer_penguins.csv
import pandas as pd
if DATA_URL:
    pen = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload palmer_penguins.csv ...")
        pen = pd.read_csv(next(iter(files.upload())))
    except Exception:
        pen = pd.read_csv("data/palmer_penguins.csv")
pen.head()

In [ ]:
PFEAT = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
print('rows before dropping missing measurements:', len(pen))
pen = pen.dropna(subset=PFEAT)
print('rows used:', len(pen))

Zp = StandardScaler().fit_transform(pen[PFEAT])
pen['cluster'] = KMeans(n_clusters=3, n_init=10, random_state=7).fit_predict(Zp)
print(pd.crosstab(pen['cluster'], pen['species']))

Each cluster is dominated by one real species, about **92%** overall. **The machinery works.**

Look at where it *does* slip. Gentoo comes out perfectly, but a handful of Adelie and Chinstrap get mixed together. That is not a bug. Those two species really are closer in body size to each other than either is to Gentoo, so the measurements alone cannot fully separate them. **The algorithm found what was in the data, and what was in the data was not quite the answer key.**

Notice two other things the toy data hid: the groups are **uneven** (Adelie 151, Gentoo 123, Chinstrap 68), and two birds had **missing measurements** and were dropped. Real data looks like this.

### Now lose the safety net

Real business data has no species column. From here on there is no answer key, and the judgement is yours. On to *Cluster the days*.

> **Where this data came from.** Gorman, K.B., Williams, T.D. & Fraser, W.R. (2014), *Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins*, PLoS ONE 9(3):e90081. Collected by the Palmer Station Antarctica LTER, a long-term climate research programme. Packaged by Horst, Presmanes Hill & Gorman (2022). Asking who collected a dataset, and why, is part of the job.